---

<div align="center">
  <img src="https://raw.githubusercontent.com/devicons/devicon/master/icons/python/python-original.svg" width="80"/>
</div>

<h1 align="center">GenAI & Advanced Nets: O Mecanismo de Atenção </h1>

<h3 align="center">PhD. Julles Mitoura</h3>

<div align="center">

  <img src="https://img.shields.io/badge/Python-3776AB?style=for-the-badge&logo=python&logoColor=white"/>
  
  <img src="https://img.shields.io/badge/Jupyter-F37626?style=for-the-badge&logo=jupyter&logoColor=white"/>

  <img src="https://img.shields.io/badge/Artificial%20Intelligence-FF6F00?style=for-the-badge&logo=openai&logoColor=white"/>

  <img src="https://img.shields.io/badge/LLM-412991?style=for-the-badge&logo=openai&logoColor=white"/>

  <img src="https://img.shields.io/badge/Transformers-FFD21E?style=for-the-badge&logo=huggingface&logoColor=black"/>

</div>

---

In [1]:
# Obs: se você não estiver utilizando um ambiente virtual, instale as bibliotecas conforme se apresenta abaixo
%pip install -q -r requirements.txt

# pip é o gerenciador de pacotes do Python. Pense nele como o instalador oficial de libs Python.
# no notebook, usar %pip ... é ideal porque instala no mesmo ambiente do kernel em uso.

# -q: quiet
# -r: requirement file, indica ao pip para instalar os pacotes listados no arquivo requirements.txt


[notice] A new release of pip available: 22.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


---

<div align="center">

## <span style="color:#1E90FF;">Anatomia da Arquitetura Transformer do Zero</span>

</div>

Na etapa anterior, desenvolvemos a arquitetura Transformer com PyTorch, aproveitando recursos prontos da biblioteca. Agora, o objetivo é implementar em Python, do zero, os principais módulos dessa arquitetura.

No modelo anterior, utilizamos o `nn.MultiheadAttention`. Embora pareça uma chamada simples, esse módulo do `torch.nn` encapsula várias operações internas. A Figura 1 ajuda a visualizar essa diferença entre a implementação pronta e a construção manual do mecanismo de atenção.

<div style="text-align: center;">
<p><em>Figura 1: Single-Head-Attention e Multi-Head Attention.</em></p>
  <img src="imgs/img03.png" alt="Arquitetura de Transformadores" width="500"/>
</div>

O objetivo agora será desenvolver em Python uma `Single-Head Attention`. Com isso, teremos mais visão sobre os detalhes desta arquitetura e poderemos seguir com o desenvolvimento.

---

Objetivo: Criar um modelo capaz de prever sequencias de comprimento de 1 a 10 tokens.

O modelo transformer será constituido por:
1. Camada de embedding
2. Mecanismo de atenção
3. Camandas Encoder e Decoder
4. Camadas Linear e Softmax

### 1. Hiperparâmetros Iniciais

In [ ]:
# imports fundamentis
import numpy as np

# dimensão do modelo
dim_model = 64
# comprimento da sequencia que será predita pelo modelo
seq_length = 10
# tamanho do vocabulario
vocab_size = 100   

#### `dim_model` (dimensão do modelo)

Define o tamanho dos vetores internos usados para representar cada token (embeddings e estados ocultos).

- **Se aumentar `dim_model`**:
  - melhora a capacidade de representação;
  - aumenta custo computacional e uso de memória;
  - pode elevar risco de overfitting em bases pequenas.
- **Se diminuir `dim_model`**:
  - reduz custo e acelera treino/inferência;
  - pode limitar a capacidade do modelo (underfitting).

#### `seq_length` (comprimento da sequência)

Define quantos tokens o modelo observa por amostra (janela de contexto).

- **Se aumentar `seq_length`**:
  - permite capturar dependências mais longas no contexto;
  - aumenta bastante o custo da atenção (aprox. quadrático em relação ao tamanho da sequência);
  - exige mais memória.
- **Se diminuir `seq_length`**:
  - torna o treino mais rápido e barato;
  - reduz o contexto disponível para predição.

#### `vocab_size` (tamanho do vocabulário)

Define quantos tokens distintos existem no espaço de entrada/saída do modelo.

- **Se aumentar `vocab_size`**:
  - melhora cobertura de tokens possíveis;
  - aumenta o tamanho da camada de saída e o custo do treinamento.
- **Se diminuir `vocab_size`**:
  - reduz custo computacional;
  - pode perder informação e aumentar ambiguidades (mais colisões de token).

### 2. Camada de Embeddings

A função de embedding tem por objetivo converter entradas sequenciais em ventores densos de tamanho fixo. Esses vetores são conhecidos como embeddings e são parte fundamental da arquiteture que estamos desenvolvendo.

In [33]:
# função para criar embeddings de uma sequência de tokens
# entrada: lista/array de IDs de token
# saída: matriz (seq_length, dim_model)

def embeddings(input_ids, vocab_size, dim_model):
    # crie uma matriz de embedding onde cada linha representa um token do vocabulário
    # a matriz é iniciada com valores aleatórios normalmente distribuidos
    embed = np.random.randn(vocab_size, dim_model)

    # para cada indice do token input, seleciona-se o embedding correspondente da matriz
    # retorna um array de embeddings correspondentes à sequencia de entrada
    return np.array([embed[i] for i in input_ids])


# exemplo de uso
input_exemplo = [3, 10, 7, 25, 1]
saida_embeddings = embeddings(input_exemplo, vocab_size=vocab_size, dim_model=dim_model)

print("Tokens de entrada:", input_exemplo)
print("Shape da saída:", saida_embeddings.shape)  # (5, 64)
print("Primeiro vetor de embedding (token 3):")
print(saida_embeddings[0])

Tokens de entrada: [3, 10, 7, 25, 1]
Shape da saída: (5, 64)
Primeiro vetor de embedding (token 3):
[-0.96217533  1.15866525 -0.29092058 -0.13392461 -0.94326606  0.76819032
  0.42327136 -0.56518884  0.25808738  1.01766951 -0.05477692 -0.68224078
  0.63534516 -1.38491055 -0.57542032  0.52546348 -0.01947024 -1.27823045
  0.2367207  -0.47279489 -0.27486648  1.17007106 -0.74887405  1.1870079
 -1.05266649 -0.70922877  0.11356797  0.11432581 -0.50059677 -0.1486514
  1.3696504  -0.85032445 -1.56654167  0.81333527 -0.76296262 -1.98590684
 -0.24777112  0.15406703  0.02118948 -0.85874857  2.15327339 -0.85067003
 -0.06270994  0.98421912 -0.00586851 -0.87434595  0.6211433  -1.06692994
  0.32299448  0.30286796  0.38884142  1.47252037  0.138464   -0.02427955
  0.18065035  0.75208883  0.21588108  0.38992639 -0.67366021 -0.65434641
  0.03205532  1.26054401  0.91548077  1.53432184]


### 3. Camada de Atenção

Buscaremos construir o mecanismo de atenção como é apresentado na Figura 2.

<div style="text-align: center;">
<p><em>Figura 2: Single-Head-Attention.</em></p>
  <img src="imgs/img04.png" alt="Arquitetura de Transformadores" width="300"/>
</div>

No mecanismo de atenção, os vetores `Q`, `K` e `V` são projeções lineares dos embeddings de entrada:

- `Q` (**Query / Consulta**): representa o que cada token está buscando no contexto;
- `K` (**Key / Chave**): representa como cada token pode ser encontrado/comparado;
- `V` (**Value / Valor**): representa o conteúdo que será combinado para formar a saída.

Antes das projeções, $X$ representa a matriz de entrada da atenção (embeddings dos tokens, geralmente já combinados com informação posicional). Se a sequência possui $n$ tokens e dimensão $d_{\text{model}}$, então:

$$
X \in \mathbb{R}^{n \times d_{\text{model}}}
$$

Formalmente:

$$
\begin{aligned}
Q &= XW_Q \\
K &= XW_K \\
V &= XW_V
\end{aligned}
$$

A atenção é calculada por:

$$
\mathrm{Attention}(Q, K, V) = \mathrm{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V
$$

em que os escores entre `Q` e `K` definem os pesos aplicados sobre `V`.